# Dataset Audit

Verify the unified dataset, view per-class instance counts, and visualize a few annotated samples.

Run this **after** `python -m scripts.prepare_dataset --raw <datas_klasoru> --out data/unified`.

In [ ]:
import sys, os
ROOT = os.path.abspath('..')
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src.dataset.audit import audit_yolo_dataset, print_audit
from src.dataset.unify import TARGET_CLASSES

DATA_ROOT = '../data/unified'
report = audit_yolo_dataset(DATA_ROOT)
print_audit(report, class_names=TARGET_CLASSES)

## Per-class distribution chart

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

splits = ['train', 'val', 'test']
counts = np.zeros((len(splits), len(TARGET_CLASSES)), dtype=int)
for si, sp in enumerate(splits):
    if sp not in report['splits']:
        continue
    for cls_id, n in report['splits'][sp]['instances_per_class'].items():
        counts[si, cls_id] = n

fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(TARGET_CLASSES))
width = 0.27
for i, sp in enumerate(splits):
    ax.bar(x + (i - 1) * width, counts[i], width, label=sp)
ax.set_xticks(x); ax.set_xticklabels(TARGET_CLASSES)
ax.set_ylabel('# instances')
ax.set_title('Per-class instance distribution')
ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

## Sample annotated images

In [ ]:
import cv2, glob, random
import matplotlib.pyplot as plt

COLORS = {0: (0, 255, 0), 1: (0, 200, 255), 2: (255, 165, 0), 3: (255, 0, 255)}

def draw(img_path):
    img = cv2.imread(img_path)[..., ::-1]
    h, w = img.shape[:2]
    lbl = img_path.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    out = img.copy()
    if os.path.exists(lbl):
        with open(lbl) as f:
            for line in f:
                parts = line.split()
                if len(parts) != 5: continue
                cls_id = int(parts[0])
                cx, cy, bw, bh = map(float, parts[1:])
                x1 = int((cx - bw/2) * w); y1 = int((cy - bh/2) * h)
                x2 = int((cx + bw/2) * w); y2 = int((cy + bh/2) * h)
                cv2.rectangle(out, (x1, y1), (x2, y2), COLORS.get(cls_id, (255,255,255)), 2)
                cv2.putText(out, TARGET_CLASSES[cls_id], (x1, max(15, y1-5)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, COLORS.get(cls_id, (255,255,255)), 2)
    return out

random.seed(0)
samples = random.sample(glob.glob(f'{DATA_ROOT}/train/images/*.jpg'), 9)
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for ax, p in zip(axes.flat, samples):
    ax.imshow(draw(p)); ax.axis('off')
    ax.set_title(os.path.basename(p)[:40], fontsize=8)
plt.tight_layout(); plt.show()